# **Organized Voila Approach**

Voila dashboard created in this notebook.

## Setup

Let's start with the imports.

In [259]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display, HTML

#package for the alluvial
import plotly.graph_objects as go

#packages for scatterplot and mapping background
import geopandas as gpd
import contextily as ctx

### Getting API Key

Each have to download our own somehow -- Elliot has

In [260]:
#THIS CELL IS COMMENTED OUT FOR TESTING PURPOSES

#pull api key
with open('carto_key.txt') as f:
    CARTO_KEY = f.read().strip()
carto_light = f"https://basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}.png?key={CARTO_KEY}"

district_gdf = gpd.read_file('PoliceDistrictDec2012_20261112.geojson')  # read geojson file

### Import and filter the dataset

In [261]:
# import dataset
df = pd.read_csv('Crimes_-_2001_to_Present_20261110.csv')

df['Date'] = df['Date'].str[:10]

# filter top categories
df['crimecol'] = df['Primary Type'].astype("category")
top_categories = df['Primary Type'].value_counts().nlargest(10).index
df.loc[~df['Primary Type'].isin(top_categories), "crimecol"] = "OTHER OFFENSE"
df.loc[df['Primary Type'].isin(top_categories), "crimecol"] = df['Primary Type']
df_lc = df[df["Year"] >= 2023].copy()
df_lc["month"] = df_lc["Date"].str[:2].astype(int)

# drop Nov and Dec of 2026 only
df_lc = df_lc[~((df_lc["Year"] == 2026) & (df_lc["month"] >= 11))]

df['month'] = df['Date'].str[:2]
df['month'] = df['month'].astype("str")
monthdict = {'01':'January','02':'February', '03':'March', '04':'April', '05':'May','06':'June', '07':'July', '08':'August','09':'September', '10':'October','11':'November','12':'December'}
df['month_name'] = df['month'].map(monthdict)

### KPIs

These are the ways we will calculate the KPIs.

Here's KPI 1

In [262]:
#KPI 1:ARREST RATE PERCENTAGE
percentage_arrests = df['Arrest'].mean() * 100
print(f"{percentage_arrests:.2f}%")

15.00%


Here's KPI 2

In [263]:
#KPI 2: Average crimes per month
year_month = df['Date'].str[6:10] + '-' + df['Date'].str[:2]
year_month = year_month[(year_month >= '2021-01') & (year_month < '2026-11')]

month_counts = year_month.value_counts()

print(f"{month_counts.mean():,.0f}")

20,109


## Let's define functions for each individual plot

The plan is to define a function for each individual plot/number we want that will be affected by the filters. Then we will define a larger function that creates the entire dashboard using those smaller functions. We will then use Voila to make the larger function display our dashboard.

...

Let's start by defining the dual pie plot:

In [264]:
def pieplotcrimes(year, n):
    pietempdf = df.query("Year == @year")
    counts = pietempdf['Primary Type'].value_counts()

    #get top n crime types from the year, get the rest as separate vars
    top = counts.head(n)
    rest = counts.iloc[n:]

    #create subplots for the two pie charts
    fig, (ax1,ax2) = plt.subplots(1,2,figsize=(20,16))

    #create main pie
    top['Other (combined)'] = rest.sum()
    top.plot.pie(ax=ax1, autopct='%1.2f%%', cmap='RdPu_r')
    ax1.set_title(f'Top {n} types of crime in {year}')
    ax1.set_ylabel('')

    #create pie of other
    rest.plot.pie(ax=ax2, autopct='%1.2f%%', cmap='RdPu_r')
    ax2.set_title(f'Breakdown of "Other" in {year}')
    ax2.set_ylabel('')

    plt.show()

Now, let's define the alluvial.

In [265]:
#new version
def alluvialplot(year):
    agg = (df.query("Year == @year")
             .groupby(['crimecol', 'Arrest', 'Domestic'], observed=True)
             .size().reset_index(name='count'))

    fig = go.Figure(go.Parcats(
        dimensions=[
            {'label': 'Crime',    'values': agg['crimecol']},
            {'label': 'Arrest',   'values': agg['Arrest']},
            {'label': 'Domestic', 'values': agg['Domestic']},
        ],
        counts=agg['count'],
        line={'color': agg['Arrest'].astype(int),
              'colorscale': [[0, 'steelblue'], [1, 'crimson']],
              'shape': 'hspline'},
    ))
    fig.update_layout(width=800, height=450)
    display(fig)

#old version
'''def alluvialplot(year):
    alluvialtempdf = df.query("Year == @year").copy()
    alluvialtempdf['ArrestNum'] = alluvialtempdf['Arrest'].astype(int)
        #alluvialtempf = df[df['Year']==year]

    fig = px.parallel_categories(
        alluvialtempdf,
        dimensions=['crimecol', 'Arrest', 'Domestic'],
        color='ArrestNum',
        color_continuous_scale=['steelblue', 'crimson'],
        width=800, height=450,
    )
    fig.update_traces(line_shape='hspline')
    fig.update_layout(coloraxis_showscale=False)
    display(fig)'''

'def alluvialplot(year):\n    alluvialtempdf = df.query("Year == @year").copy()\n    alluvialtempdf[\'ArrestNum\'] = alluvialtempdf[\'Arrest\'].astype(int)\n        #alluvialtempf = df[df[\'Year\']==year]\n\n    fig = px.parallel_categories(\n        alluvialtempdf,\n        dimensions=[\'crimecol\', \'Arrest\', \'Domestic\'],\n        color=\'ArrestNum\',\n        color_continuous_scale=[\'steelblue\', \'crimson\'],\n        width=800, height=450,\n    )\n    fig.update_traces(line_shape=\'hspline\')\n    fig.update_layout(coloraxis_showscale=False)\n    display(fig)'

Now, let's define the stacked bar chart.

In [266]:
def bar_plot(year):
    barplotdf = df.query("Year == @year").copy()
    g = (barplotdf.groupby('Primary Type')['Arrest']
            .agg(total='size', arrests='sum')
            .query('total >= 500')
    )
    
    #g = g.sort_values('total')
    g['rate'] = g['arrests'] / g['total']
    g = g.sort_values('rate')
    g['total'].plot.barh(color='steelblue', figsize=(8, 8), label='Total incidents')
    g['arrests'].plot.barh(color='crimson', label='Arrests')
    plt.xlabel('Number of incidents')
    plt.legend()
    plt.show()

Now, let's define the scatterplot of crime location.

In [267]:
def scatterplot(year, crime, district):
    scatterplot_df = df.query("Year == @year and crimecol == @crime")
    if district == "All":   # whole city: right panel shows every district
        scatterplot_district_df = scatterplot_df
        selected = district_gdf
    else:
        scatterplot_district_df = scatterplot_df[scatterplot_df['District'] == district]
        selected = district_gdf[district_gdf['dist_num'].astype(int) == district]

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 7))

    district_gdf.boundary.plot(ax=ax0, color='grey', linewidth=0.5)
    selected.boundary.plot(ax=ax0, color='black', linewidth=2)   # show where the district is
    scatterplot_df.plot.scatter(ax=ax0, x='Longitude', y='Latitude', s=0.7, alpha=.08,
                                color='crimson')
    minx, miny, maxx, maxy = district_gdf.total_bounds              # frame the whole city
    ax0.set_xlim(minx - 0.01, maxx + 0.01)
    ax0.set_ylim(miny - 0.01, maxy + 0.01)
    ctx.add_basemap(ax0, crs='EPSG:4326', source=carto_light)
    ax0.set_title(f'Map of {crime} in Chicago in {year}')
    ax0.set_axis_off()

    district_gdf.boundary.plot(ax=ax1, color='grey', linewidth=0.6)
    selected.boundary.plot(ax=ax1, color='black', linewidth=2)
    scatterplot_district_df.plot.scatter(ax=ax1, x='Longitude', y='Latitude', s=1.2, alpha=0.5,
                                             color='crimson')
    minx, miny, maxx, maxy = selected.total_bounds
    ax1.set_xlim(minx - 0.01, maxx + 0.01)
    ax1.set_ylim(miny - 0.01, maxy + 0.01)
    ctx.add_basemap(ax1, crs='EPSG:4326', source=carto_light)
    ax1.set_title(f'Map of {crime} in all districts in {year}' if district == "All" else f'Map of {crime} in District {district} in {year}')
    ax1.set_axis_off()

    plt.tight_layout()
    plt.show()   # only one show, at the end, so both maps render in the same figure
    attribution='© OpenStreetMap contributors © CARTO'

Now, let's define the line plot.

In [268]:
# first, a small function needed for the line plot creation function
def monthly_counts(crime, district="All"):
    d = df_lc[df_lc["Primary Type"] == crime]
    if district != "All":
        d = d[d["District"] == district]
    return d.groupby(["Year", "month"]).size().reset_index(name="count")

# now, the line plot creation function
def plot_line(crime, district):
    counts = monthly_counts(crime, district)
    title = f"{crime.title()} per Month by Year"
    if district != "All":
        title += f" (District {district})"

    month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    fig = px.line(counts, x="month", y="count", color="Year",
                  markers=True, title=title)
    fig.update_xaxes(tickmode="array", tickvals=list(range(1, 13)),
                     ticktext=month_labels, title="Month")
    fig.update_yaxes(title="Number of incidents")
    fig.update_layout(legend_title="Year")
    display(fig)

Now, let's define the matrix plot.

In [269]:
def matrixplot(crime):
    matrixplot = df.query("crimecol == @crime")
    df_wide = matrixplot.pivot_table(index='month_name', columns='Year', values='ID', aggfunc='count')
    df_wide = df_wide.reindex(["January", "February", "March","April","May","June","July","August","September","October","November","December"])
    # need to set the dec and nov of 26 as 0, or like a null, but the 0 is a temp placeholder
    df_wide=df_wide.fillna(0)
    df_wide=df_wide.astype('int')

    other_years = [c for c in df_wide.columns if c != 2026]

    styled = (
        df_wide.style
        .background_gradient(cmap='PuRd', subset=other_years)
        .background_gradient(cmap='PuRd', subset=[2026], vmin=3657, vmax=4640)
    )

    display(styled)

### Let's define the master output function

We want to wrap each of our individual plots into one major function that we will use for the Voila function.

In [284]:
def plot_dashboard(year, n, crime, district):
    pieplotcrimes(year=year, n=n) #d

    alluvialplot(year=year) #i

    bar_plot(year=year) #e (make it out of 1 instead of having scale)

    matrixplot(crime=crime) #g

    plot_line(crime=crime, district=district) #h

    scatterplot(year=year,crime=crime,district=district) #f

## Let's create the voila output

We will create the voila output by defining controls and the output, then using the display() function to create our dashboard.

### Let's start by defining our interactive options.

Firstly, the year dropdown:

In [278]:
year_dropdown = widgets.Dropdown(
    options=[2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026],
    value=2026,
    description='Year:'
)

Next, the slider for n (number of crimes before otherizing everything)

In [279]:
n_slider = widgets.IntSlider(
    value=8,
    min=1,
    max=14,
    step=1,
    description='Top N:'
)

Next, the crime type dropdown

In [280]:
crimes = sorted(df_lc["Primary Type"].dropna().unique())

crime_dropdown = widgets.Dropdown(options=crimes, value="THEFT", description="crime")

Finally, the district number dropdown

In [281]:
district_options = ["All"] + sorted(df_lc["District"].dropna().unique().tolist())

district_dropdown = widgets.Dropdown(options=district_options, value=1, description="district")

### Next, let's set our controls and output variables

In [ ]:
controls = widgets.HBox([year_dropdown, n_slider, crime_dropdown, district_dropdown])
output = widgets.Output()
load_button = widgets.Button(description="Load dashboard", button_style="primary")

def update(change=None):
    with output:
        output.clear_output(wait=True)
        plot_dashboard(year_dropdown.value, n_slider.value, crime_dropdown.value, district_dropdown.value)

load_button.on_click(update)
for w in (year_dropdown, n_slider, crime_dropdown, district_dropdown):
    w.observe(update, names="value")

### The Grand Reveal: Plotting the Actual Dashboard

In [283]:
display(controls, load_button, output)

Button(button_style='primary', description='Load dashboard', style=ButtonStyle())

Output()